<a href="https://colab.research.google.com/github/SRIJANRAOS/srijanraos_INFO5731_spring2026/blob/main/In_Class_Exercise_Combined_ipynb_(1)_(1)_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Exercise: Sentiment Analysis + Text Classification + Prompt Engineering

**Course:** NLP / Text Mining  
**Total Points:** 30  
**Suggested Time:** about 45 minutes total  
**Dataset:** `sentiment_classification_combined_data.csv`

**Instructions**
- Complete all questions in this notebook.
- Run all cells before submission.
- Keep your answers clear and organized.
- For the prompt engineering part, you do **not** need an API key.


## Part A — Sentiment Analysis with VADER (10 points)
In this part, use **VADER** to compute sentiment scores for the text data.

In [1]:
# Upload the dataset file from your computer (Google Colab)
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
print("Uploaded file name:", filename)


Saving sentiment_classification_combined_data.csv to sentiment_classification_combined_data.csv
Uploaded file name: sentiment_classification_combined_data.csv


### Q1. Load and inspect the dataset (2 points)
Show:
1. the first 5 rows  
2. the shape of the dataset  
3. the column names

In [2]:
import pandas as pd

df = pd.read_csv(filename)

# 1. First 5 rows
print("First 5 rows:")
display(df.head())

# 2. Shape of the dataset
print("Shape:", df.shape)

# 3. Column names
print("Columns:", df.columns.tolist())


First 5 rows:


,Text,TrueLabel,Score
0,I loved this healthcare AI workshop. The examp...,positive,5
1,"This lecture was okay, but some parts were con...",neutral,3
2,The notebook was easy to follow and very pract...,positive,5
3,I did not like the explanation of the model re...,negative,1
4,The activity was interesting and helped me und...,positive,4


Shape: (24, 3)
Columns: ['Text', 'TrueLabel', 'Score']


### Q2. Apply VADER to compute sentiment scores (3 points)
- Import VADER
- Download the lexicon if needed
- Create a new column called `compound`
- Show the first 5 rows of `Text` and `compound`

In [3]:
!pip -q install nltk
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [4]:
# Apply VADER to each text and store the compound score
df['compound'] = df['Text'].apply(lambda text: sia.polarity_scores(text)['compound'])

# Show first 5 rows of Text and compound
print(df[['Text', 'compound']].head())

                                                Text  compound
0  I loved this healthcare AI workshop. The examp...    0.8555
1  This lecture was okay, but some parts were con...   -0.2263
2  The notebook was easy to follow and very pract...    0.4404
3  I did not like the explanation of the model re...   -0.2755
4  The activity was interesting and helped me und...    0.4019


### Q3. Create a sentiment label from the compound score (3 points)
Create a new column called `PredictedSentiment` using this rule:
- `compound >= 0.05` → `positive`
- `compound <= -0.05` → `negative`
- otherwise → `neutral`

Then show the first 10 rows of `Text`, `compound`, and `PredictedSentiment`.

In [5]:
# Define a function to convert compound score to a sentiment label
def get_sentiment_label(compound_score):
    if compound_score >= 0.05:
        return 'positive'
    elif compound_score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

# Apply the function to create the PredictedSentiment column
df['PredictedSentiment'] = df['compound'].apply(get_sentiment_label)

# Show first 10 rows of the relevant columns
print(df[['Text', 'compound', 'PredictedSentiment']].head(10).to_string())


                                                                        Text  compound PredictedSentiment
0   I loved this healthcare AI workshop. The examples were clear and useful.    0.8555           positive
1                      This lecture was okay, but some parts were confusing.   -0.2263           negative
2                        The notebook was easy to follow and very practical.    0.4404           positive
3                       I did not like the explanation of the model results.   -0.2755           negative
4  The activity was interesting and helped me understand sentiment analysis.    0.4019           positive
5            The class was too fast and I could not follow the coding steps.    0.0000            neutral
6       The topic is important, but the instructions were not very detailed.    0.1027           positive
7                    I really enjoyed the examples and the live coding demo.    0.5563           positive
8                       The dataset was messy 

### Q4. Compare review score and sentiment (2 points)
Group by `Score` and compute the **average compound score** for each score value.

In [6]:
# Group by Score and compute average compound score
avg_compound_by_score = df.groupby('Score')['compound'].mean().reset_index()
avg_compound_by_score.columns = ['Score', 'Average Compound Score']
avg_compound_by_score = avg_compound_by_score.sort_values('Score')

print("Average VADER Compound Score by Review Score:")
print(avg_compound_by_score.to_string(index=False))


Average VADER Compound Score by Review Score:
 Score  Average Compound Score
     1               -0.507033
     2               -0.296125
     3               -0.150783
     4                0.478340
     5                0.621167


## Part B — Prompt Engineering (10 points)
This part is based on the **Prompt Engineering** notebook. You do **not** need an API key. Focus on writing better prompts using the concepts of **instruction, context, output format, zero-shot, few-shot, and chain-of-thought**.

In [7]:
# Sample texts for the prompt engineering questions
sample_reviews = [
    "The product arrived late and the quality was poor.",
    "Amazing battery life and elegant design.",
    "It is fine for daily use, nothing special."
]
for i, text in enumerate(sample_reviews, start=1):
    print(f"{i}. {text}")

1. The product arrived late and the quality was poor.
2. Amazing battery life and elegant design.
3. It is fine for daily use, nothing special.


### Q1. Zero-shot classification prompt (3 points)
Write a **zero-shot prompt** that asks an AI model to classify each review as **positive, negative, or neutral**.

Your prompt should include:
- a clear instruction
- the three labels
- a request for a clean output format

Store your prompt in a Python variable called `zero_shot_prompt`, then print it.

In [9]:
zero_shot_prompt = """
You are a sentiment analysis assistant. Your task is to classify each of the
following product reviews as exactly one of three labels: positive, negative,
or neutral.

Rules:
- Use 'positive' if the review expresses satisfaction, praise, or approval.
- Use 'negative' if the review expresses dissatisfaction, criticism, or disappointment.
- Use 'neutral' if the review is neither clearly positive nor clearly negative.

Output format: For each review, respond on a new line using exactly this format:
Review <number>: <label>

Reviews to classify:
1. The product arrived late and the quality was poor.
2. Amazing battery life and elegant design.
3. It is fine for daily use, nothing special.
"""

print(zero_shot_prompt)


You are a sentiment analysis assistant. Your task is to classify each of the
following product reviews as exactly one of three labels: positive, negative,
or neutral.

Rules:
- Use 'positive' if the review expresses satisfaction, praise, or approval.
- Use 'negative' if the review expresses dissatisfaction, criticism, or disappointment.
- Use 'neutral' if the review is neither clearly positive nor clearly negative.

Output format: For each review, respond on a new line using exactly this format:
Review <number>: <label>

Reviews to classify:
1. The product arrived late and the quality was poor.
2. Amazing battery life and elegant design.
3. It is fine for daily use, nothing special.



### Q2. Few-shot prompt improvement (3 points)
Now improve the previous prompt by adding **two examples**.

Requirements:
- Store it in a variable called `few_shot_prompt`
- Include 2 input-output examples
- Ask the model to classify the 3 sample reviews after the examples

In [10]:
few_shot_prompt = """
You are a sentiment analysis assistant. Your task is to classify product reviews
as positive, negative, or neutral.

Here are two examples to guide your classification:

Example 1:
Review: "This is the best purchase I have ever made. Works perfectly!"
Label: positive

Example 2:
Review: "The packaging was damaged and the item stopped working after two days."
Label: negative

Now classify each of the following reviews using the same format.
Output format: For each review, respond on a new line as:
Review <number>: <label>

Reviews to classify:
1. The product arrived late and the quality was poor.
2. Amazing battery life and elegant design.
3. It is fine for daily use, nothing special.
"""

print(few_shot_prompt)


You are a sentiment analysis assistant. Your task is to classify product reviews
as positive, negative, or neutral.

Here are two examples to guide your classification:

Example 1:
Review: "This is the best purchase I have ever made. Works perfectly!"
Label: positive

Example 2:
Review: "The packaging was damaged and the item stopped working after two days."
Label: negative

Now classify each of the following reviews using the same format.
Output format: For each review, respond on a new line as:
Review <number>: <label>

Reviews to classify:
1. The product arrived late and the quality was poor.
2. Amazing battery life and elegant design.
3. It is fine for daily use, nothing special.



### Q3. Prompt comparison and refinement (4 points)
Create a short table or DataFrame with **three columns**:
- `PromptType`
- `MainFeature`
- `WhyUseful`

Include these three prompt types:
1. Zero-shot
2. Few-shot
3. Chain-of-thought

Then, in **2–3 sentences**, explain which one you would choose for a more difficult text analysis task and why.

In [11]:
import pandas as pd

prompt_comparison = pd.DataFrame({
    'PromptType': ['Zero-shot', 'Few-shot', 'Chain-of-thought'],
    'MainFeature': [
        'No examples provided; relies entirely on the model\'s pre-trained knowledge',
        'Includes labeled input-output examples to demonstrate the expected behavior',
        'Asks the model to reason through each step before producing the final answer'
    ],
    'WhyUseful': [
        'Quick and simple for straightforward tasks where the model already understands the domain well',
        'Improves accuracy for ambiguous or domain-specific tasks by showing the model what good output looks like',
        'Best for complex tasks (e.g., multi-label classification, argument mining) where intermediate reasoning reduces errors'
    ]
})

print(prompt_comparison.to_string(index=False))

print("""
--- Written Explanation ---
For a more difficult text analysis task, such as detecting sarcasm or classifying
nuanced opinion in long documents, I would choose chain-of-thought prompting.
Unlike zero-shot or few-shot approaches, chain-of-thought explicitly asks the model
to reason step by step before committing to a label, which reduces errors caused by
surface-level pattern matching. This is especially valuable when the correct label
depends on understanding context, tone, or implicit meaning rather than obvious keywords.
""")


      PromptType                                                                  MainFeature                                                                                                              WhyUseful
       Zero-shot   No examples provided; relies entirely on the model's pre-trained knowledge                         Quick and simple for straightforward tasks where the model already understands the domain well
        Few-shot  Includes labeled input-output examples to demonstrate the expected behavior              Improves accuracy for ambiguous or domain-specific tasks by showing the model what good output looks like
Chain-of-thought Asks the model to reason through each step before producing the final answer Best for complex tasks (e.g., multi-label classification, argument mining) where intermediate reasoning reduces errors

--- Written Explanation ---
For a more difficult text analysis task, such as detecting sarcasm or classifying
nuanced opinion in long documents, I 

## Reflection (optional, no extra points)
Briefly explain the difference between:
- sentiment analysis
- text classification
- prompt engineering